# LangChain Loop / ReAct 快速入门

当前 notebook 用来验证最小显式 loop / ReAct 链路：

1. 读取并校验环境变量
2. 运行普通问题，观察直接 final
3. 运行天气问题，观察 tool -> final
4. 运行缺少城市的问题，观察直接追问
5. 查看 steps、step_count、stop_reason
6. 对比 03_agent 的隐式 tool calling 与当前显式 loop / ReAct


In [15]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.config.env import load_env
from src.chains.agent_chain import run_weather_agent
from src.chains.loop_react_chain import run_weather_loop_react


In [16]:
env = load_env()
print({
    "ollama_base_url": env.ollama_base_url,
    "ollama_model": env.ollama_model,
    "weather_api_key_configured": bool(env.weather_api_key),
})


{'ollama_base_url': 'http://localhost:11434', 'ollama_model': 'qwen3:8b', 'weather_api_key_configured': True}


In [17]:
final_result = run_weather_loop_react("请用一句话解释什么是 LangChain")
print(final_result)


{'mode': 'loop-react', 'input': '请用一句话解释什么是 LangChain', 'used_tools': False, 'tool_names': [], 'steps': [{'step': 1, 'decision_type': 'final', 'reason': '用户的问题是关于LangChain的定义，不需要天气工具', 'raw_model_output': '{\n  "type": "final",\n  "reason": "用户的问题是关于LangChain的定义，不需要天气工具",\n  "answer": "LangChain是一个用于构建端到端AI应用的框架，提供工具调用、记忆管理和提示工程等功能"\n}', 'tool_name': None, 'tool_input': None, 'tool_output': None, 'tool_status': None, 'final_answer': 'LangChain是一个用于构建端到端AI应用的框架，提供工具调用、记忆管理和提示工程等功能'}], 'step_count': 1, 'stop_reason': 'final_answer', 'final_answer': 'LangChain是一个用于构建端到端AI应用的框架，提供工具调用、记忆管理和提示工程等功能'}


In [18]:
tool_result = run_weather_loop_react("北京现在天气怎么样？")
print(tool_result)


{'mode': 'loop-react', 'input': '北京现在天气怎么样？', 'used_tools': True, 'tool_names': ['get_weather'], 'steps': [{'step': 1, 'decision_type': 'tool', 'reason': '问题涉及天气且明确给出了城市名称北京，需要调用get_weather工具获取天气信息', 'raw_model_output': '{\n  "type": "tool",\n  "reason": "问题涉及天气且明确给出了城市名称北京，需要调用get_weather工具获取天气信息",\n  "tool_name": "get_weather",\n  "tool_input": {\n    "city": "北京"\n  }\n}', 'tool_name': 'get_weather', 'tool_input': {'city': '北京'}, 'tool_output': '北京，Beijing，中国；当地时间 2026-03-30 11:18；天气 晴天；当前温度 15.3°C；体感温度 15.3°C；湿度 21%；风速 7.2 km/h。', 'tool_status': 'success', 'final_answer': None}, {'step': 2, 'decision_type': 'final', 'reason': '最近一步已通过get_weather获取北京天气数据，可直接输出结果', 'raw_model_output': '{\n  "type": "final",\n  "reason": "最近一步已通过get_weather获取北京天气数据，可直接输出结果",\n  "answer": "北京当前天气晴天，温度15.3°C，湿度21%，风速7.2 km/h"\n}', 'tool_name': None, 'tool_input': None, 'tool_output': None, 'tool_status': None, 'final_answer': '北京当前天气晴天，温度15.3°C，湿度21%，风速7.2 km/h'}], 'step_count': 2, 'stop_reason': 'final

In [19]:
missing_city_result = run_weather_loop_react("现在天气怎么样？")
print(missing_city_result)


{'mode': 'loop-react', 'input': '现在天气怎么样？', 'used_tools': False, 'tool_names': [], 'steps': [{'step': 1, 'decision_type': 'final', 'reason': '问题天气相关但未提供城市名', 'raw_model_output': '{\n  "type": "final",\n  "reason": "问题天气相关但未提供城市名",\n  "answer": "请提供城市名称以便查询天气"\n}', 'tool_name': None, 'tool_input': None, 'tool_output': None, 'tool_status': None, 'final_answer': '请提供城市名称以便查询天气'}], 'step_count': 1, 'stop_reason': 'final_answer', 'final_answer': '请提供城市名称以便查询天气'}


In [20]:
print(tool_result["steps"])
print(tool_result["step_count"])
print(tool_result["stop_reason"])


[{'step': 1, 'decision_type': 'tool', 'reason': '问题涉及天气且明确给出了城市名称北京，需要调用get_weather工具获取天气信息', 'raw_model_output': '{\n  "type": "tool",\n  "reason": "问题涉及天气且明确给出了城市名称北京，需要调用get_weather工具获取天气信息",\n  "tool_name": "get_weather",\n  "tool_input": {\n    "city": "北京"\n  }\n}', 'tool_name': 'get_weather', 'tool_input': {'city': '北京'}, 'tool_output': '北京，Beijing，中国；当地时间 2026-03-30 11:18；天气 晴天；当前温度 15.3°C；体感温度 15.3°C；湿度 21%；风速 7.2 km/h。', 'tool_status': 'success', 'final_answer': None}, {'step': 2, 'decision_type': 'final', 'reason': '最近一步已通过get_weather获取北京天气数据，可直接输出结果', 'raw_model_output': '{\n  "type": "final",\n  "reason": "最近一步已通过get_weather获取北京天气数据，可直接输出结果",\n  "answer": "北京当前天气晴天，温度15.3°C，湿度21%，风速7.2 km/h"\n}', 'tool_name': None, 'tool_input': None, 'tool_output': None, 'tool_status': None, 'final_answer': '北京当前天气晴天，温度15.3°C，湿度21%，风速7.2 km/h'}]
2
final_answer


In [21]:
print(run_weather_agent("北京现在天气怎么样？"))
print(run_weather_loop_react("北京现在天气怎么样？"))


{'input': '北京现在天气怎么样？', 'used_tools': True, 'tool_names': ['get_weather'], 'intermediate_steps': [{'round': 1, 'tool_name': 'get_weather', 'tool_args': {'city': '北京'}, 'tool_output': '北京，Beijing，中国；当地时间 2026-03-30 11:18；天气 晴天；当前温度 15.3°C；体感温度 15.3°C；湿度 21%；风速 7.2 km/h。'}], 'final_answer': '北京现在是晴天，当前温度15.3°C，体感温度同样为15.3°C，空气湿度21%，风速7.2 km/h。天气条件较为舒适，适合户外活动。'}
{'mode': 'loop-react', 'input': '北京现在天气怎么样？', 'used_tools': True, 'tool_names': ['get_weather'], 'steps': [{'step': 1, 'decision_type': 'tool', 'reason': '问题询问北京天气，已明确给出城市名，需要调用get_weather工具', 'raw_model_output': '{\n  "type": "tool",\n  "reason": "问题询问北京天气，已明确给出城市名，需要调用get_weather工具",\n  "tool_name": "get_weather",\n  "tool_input": {\n    "city": "北京"\n  }\n}', 'tool_name': 'get_weather', 'tool_input': {'city': '北京'}, 'tool_output': '北京，Beijing，中国；当地时间 2026-03-30 11:18；天气 晴天；当前温度 15.3°C；体感温度 15.3°C；湿度 21%；风速 7.2 km/h。', 'tool_status': 'success', 'final_answer': None}, {'step': 2, 'decision_type': 'final', 'reason': '最近一步已通过get_we